In [83]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
from matplotlib.figure import Figure
import mlflow
from mlflow.tracking import MlflowClient
from datetime import datetime

def debug(msg: str) -> None:
    """Log messages with timestamp."""
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}")

def plot_global_losses(history: dict, domain: str, anomaly: str) -> Figure:
    fig = Figure(figsize=(9, 5))
    ax = fig.add_subplot(1, 1, 1)
    
    ax.plot(history["g_loss"], label="Generator Loss", alpha=0.8)
    ax.plot(history["d_loss"], label="Discriminator Loss", alpha=0.8)
    ax.set_title(f"Global Losses - {domain} - {anomaly}")
    ax.set_xlabel("Steps")
    ax.legend()
    ax.grid(True, alpha=0.3)

    fig.tight_layout()
    fig.savefig(f"../data/08_reporting/plots_losses/losses_{domain}_{anomaly}.pdf")
    return fig

In [22]:
history = catalog.load("training.training_history")

current_dir = os.getcwd()
root_dir = os.path.dirname(current_dir)

mlflow.set_tracking_uri(
    "sqlite:///C:/Users/tmdp1/masters-dissertation-data-science/mlruns.db"
)

client = MlflowClient()

experiment = client.get_experiment_by_name("masters_dissertation_data_science")

runs = client.search_runs(experiment_ids=[experiment.experiment_id])

[09/09/26 00:14:47] INFO     Loading data from training.training_history                       ]8;id=14928595;file://C:\Users\tmdp1\masters-dissertation-data-science\.venv\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=14928596;file://C:\Users\tmdp1\masters-dissertation-data-science\.venv\Lib\site-packages\kedro\io\data_catalog.py#1050\1050]8;;\
                             (MlflowPickleDataset)...                                                              

In [76]:
df_metrics_all = pd.DataFrame({
    "d_loss": [],
    "step": [],
    "g_loss": [],
    "d_real": [],
    "d_fake": [],
    "category": []
})

total_runs = len(runs)
for i, run in enumerate(runs, start=1):
    debug(f"Run {i}/{total_runs}")
    run_id = run.info.run_id
    run_name = run.data.tags.get("mlflow.runName", "default_run")
    
    metric_names = run.data.metrics.keys() 
    
    metrics_data = {}
    
    for metric_name in metric_names:
        history = client.get_metric_history(run_id, metric_name)
        metrics_data[metric_name] = [m.value for m in history]
        
        if "step" not in metrics_data:
            metrics_data["step"] = [m.step for m in history]
    
    df_metrics = pd.DataFrame(metrics_data)
    
    df_metrics = df_metrics.sort_values(by="step").reset_index(drop=True)
    
    df_metrics["category"] = run_name
    df_metrics["run_id"] = run_id
    
    df_metrics_all = pd.concat([df_metrics_all, df_metrics], ignore_index=True)

df_metrics_all.to_csv("../data/08_reporting/metrics.csv", index=False)


[2026-09-09 00:55:34] Run 1/42
[2026-09-09 00:56:04] Run 2/42
[2026-09-09 00:56:11] Run 3/42
[2026-09-09 00:56:19] Run 4/42
[2026-09-09 00:56:25] Run 5/42
[2026-09-09 00:56:31] Run 6/42
[2026-09-09 00:56:38] Run 7/42
[2026-09-09 00:56:45] Run 8/42
[2026-09-09 00:56:52] Run 9/42
[2026-09-09 00:56:59] Run 10/42
[2026-09-09 00:57:11] Run 11/42
[2026-09-09 00:57:21] Run 12/42
[2026-09-09 00:57:30] Run 13/42
[2026-09-09 00:57:39] Run 14/42
[2026-09-09 00:57:51] Run 15/42
[2026-09-09 00:57:54] Run 16/42
[2026-09-09 00:57:58] Run 17/42
[2026-09-09 00:58:02] Run 18/42
[2026-09-09 00:58:06] Run 19/42
[2026-09-09 00:58:09] Run 20/42
[2026-09-09 00:58:12] Run 21/42
[2026-09-09 00:58:15] Run 22/42
[2026-09-09 00:58:18] Run 23/42
[2026-09-09 00:58:24] Run 24/42
[2026-09-09 00:58:34] Run 25/42
[2026-09-09 00:58:45] Run 26/42
[2026-09-09 00:58:55] Run 27/42
[2026-09-09 00:59:04] Run 28/42
[2026-09-09 00:59:10] Run 29/42
[2026-09-09 00:59:14] Run 30/42
[2026-09-09 00:59:19] Run 31/42
[2026-09-09 00:59

In [85]:
df_metrics_all = pd.read_csv("../data/08_reporting/metrics.csv")

all_cats = df_metrics_all.category.unique()

for cat in all_cats:
    tmp = cat.split("_")
    domain = tmp[0]
    anomaly = tmp[1]
    
    metrics = df_metrics_all[df_metrics_all["category"] == cat]
    
    history =  {}
    history["g_loss"] = metrics.g_loss.values
    history["d_loss"] = metrics.d_loss.values
    fig = plot_global_losses(history, domain, anomaly)

In [39]:
history = {}

history["g_loss"] = pd.read_csv("../data/08_reporting/g_loss.csv").value.values
history["g_loss"] = np.array([x.replace("'", "") for x in history["g_loss"]], dtype=float)
history["d_real"] = pd.read_csv("../data/08_reporting/d_real.csv").value.values
history["d_real"] = np.array([x.replace("'", "") for x in history["d_real"]], dtype=float)
history["d_loss"] = pd.read_csv("../data/08_reporting/d_loss.csv").value.values
history["d_loss"] = np.array([x.replace("'", "") for x in history["d_loss"]], dtype=float)
history["d_fake"] = pd.read_csv("../data/08_reporting/d_fake.csv").value.values
history["d_fake"] = np.array([x.replace("'", "") for x in history["d_fake"]], dtype=float)
history["g_loss_d"] = pd.read_csv("../data/08_reporting/g_loss_d.csv").value.values
history["g_loss_d"] = np.array([x.replace("'", "") for x in history["g_loss_d"]], dtype=float)

In [5]:
import pandas as pd

file_path = "C:/Users/tmdp1/masters-dissertation-data-science/mlruns/1/87a877067ba14a4eb63918c6f305eef5/artifacts/metrics/training_history.pkl"

training_history = pd.read_pickle(file_path)
training_history.keys()
len(training_history["d_loss"])

300

In [41]:
fig1 = plot_global_losses(history)
fig1.savefig("../data/08_reporting/global_losses.pdf", format="pdf")

fig2 = plot_critic_scores(history)
fig2.savefig("../data/08_reporting/critic_scores.pdf", format="pdf")